# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

## Cell 1 – Imports

In [1]:
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

## Cell 2 – Configuration & Clients

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

GPT_MODEL    = "gpt-4.1-mini"
OLLAMA_MODEL = "llama3.2"
DB           = "tech_qa.db"

openai_client = OpenAI()
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

print(f"GPT model:   {GPT_MODEL}")
print(f"Ollama model: {OLLAMA_MODEL}")

OpenAI API Key exists and begins sk-proj-
GPT model:   gpt-4.1-mini
Ollama model: llama3.2


## Cell 3 – SQLite Database Init

We create a persistent **code snippet library** that the assistant can save to and retrieve from.
This demonstrates tool calling with a real database backend (same pattern as Day 4).

In [3]:
def init_db():
    with sqlite3.connect(DB) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS snippets (
                name        TEXT PRIMARY KEY,
                code        TEXT NOT NULL,
                description TEXT DEFAULT ''
            )
        """)
        conn.commit()

init_db()
print(f"Database '{DB}' ready.")

Database 'tech_qa.db' ready.


## Cell 4 – Tool Functions

Three functions backed by SQLite:
- `save_snippet` – store a named code snippet
- `get_snippet`  – retrieve a snippet by name
- `list_snippets` – list all saved snippet names and descriptions

In [4]:
def save_snippet(name: str, code: str, description: str = "") -> str:
    print(f"TOOL: save_snippet(name={name!r})", flush=True)
    with sqlite3.connect(DB) as conn:
        conn.execute(
            "INSERT INTO snippets (name, code, description) VALUES (?, ?, ?)"
            " ON CONFLICT(name) DO UPDATE SET code=excluded.code, description=excluded.description",
            (name.lower(), code, description)
        )
        conn.commit()
    return f"Snippet '{name}' saved successfully."


def get_snippet(name: str) -> str:
    print(f"TOOL: get_snippet(name={name!r})", flush=True)
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT code, description FROM snippets WHERE name = ?",
            (name.lower(),)
        ).fetchone()
    if row:
        code, desc = row
        result = f"**{name}**"
        if desc:
            result += f" – {desc}"
        result += f"\n```python\n{code}\n```"
        return result
    return f"No snippet found with name '{name}'."


def list_snippets() -> str:
    print("TOOL: list_snippets()", flush=True)
    with sqlite3.connect(DB) as conn:
        rows = conn.execute(
            "SELECT name, description FROM snippets ORDER BY name"
        ).fetchall()
    if not rows:
        return "No snippets saved yet."
    lines = [f"- **{name}**: {desc if desc else '(no description)'}" for name, desc in rows]
    return "Saved snippets:\n" + "\n".join(lines)


print("Tool functions ready.")

Tool functions ready.


## Cell 5 – Tool Schemas & Dispatch Registry

In [5]:
save_snippet_function = {
    "name": "save_snippet",
    "description": "Save a named Python code snippet to the persistent library for later retrieval.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "Short unique identifier for the snippet (e.g. 'list_comp', 'fizzbuzz')"
            },
            "code": {
                "type": "string",
                "description": "The Python code to save"
            },
            "description": {
                "type": "string",
                "description": "Optional one-line description of what the snippet does"
            }
        },
        "required": ["name", "code"],
        "additionalProperties": False
    }
}

get_snippet_function = {
    "name": "get_snippet",
    "description": "Retrieve a previously saved Python code snippet by name.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "The name of the snippet to retrieve"
            }
        },
        "required": ["name"],
        "additionalProperties": False
    }
}

list_snippets_function = {
    "name": "list_snippets",
    "description": "List all saved Python code snippets with their names and descriptions.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": save_snippet_function},
    {"type": "function", "function": get_snippet_function},
    {"type": "function", "function": list_snippets_function},
]

TOOL_HANDLERS = {
    "save_snippet":  lambda a: save_snippet(a["name"], a["code"], a.get("description", "")),
    "get_snippet":   lambda a: get_snippet(a["name"]),
    "list_snippets": lambda a: list_snippets(),
}

print(f"{len(tools)} tools registered: {[t['function']['name'] for t in tools]}")

3 tools registered: ['save_snippet', 'get_snippet', 'list_snippets']


## Cell 6 – System Prompt

An expert Python & programming tutor persona.
We also tell the model about its snippet tools so it knows when to use them.

In [17]:
system_message = """
You are an expert Python and programming tutor for Sebastian Gil with deep knowledge of:
- Python fundamentals, idioms, and best practices
- Data structures, algorithms, and design patterns
- The Python standard library and popular packages
- Software engineering principles

Your style:
- Give clear, accurate explanations with concrete examples
- Always use markdown formatting with fenced code blocks (```python) for code
- Break down complex topics step by step
- Point out common pitfalls and how to avoid them
- Be encouraging but precise

You have access to a persistent code snippet library. Use it proactively:
- When you demonstrate useful code, offer to save it with `save_snippet`
- When a user asks to save a snippet, use `save_snippet` immediately
- When asked what snippets are available, use `list_snippets`
- When asked to show a snippet, use `get_snippet`

Tip for users: you can say things like:
  "Save that as 'decorator_example'"
  "Show me my fizzbuzz snippet"
  "What snippets do I have saved?"
""".strip()

print("System prompt set.")

System prompt set.


## Cell 7 – handle_tool_calls()

Dispatch-registry pattern from Day 4: handles multiple tool calls in one response.

In [7]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        name      = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        try:
            handler = TOOL_HANDLERS.get(name)
            result  = handler(arguments) if handler else f"Unknown tool: {name}"
        except Exception as e:
            result = f"Error calling {name}: {e}"
        responses.append({
            "role":        "tool",
            "content":     str(result),
            "tool_call_id": tool_call.id,
        })
    return responses

print("handle_tool_calls() ready.")

handle_tool_calls() ready.


## Cell 8 – chat() Callback

Combines:
- Model switching (GPT vs Ollama)
- Tool calling loop (GPT only; Ollama tool support is unreliable)
- Streaming final response via `yield`

In [8]:
def chat(message, history, model):
    use_gpt   = "GPT" in model
    client    = openai_client if use_gpt else ollama_client
    model_id  = GPT_MODEL     if use_gpt else OLLAMA_MODEL
    use_tools = use_gpt  # Only use tools with OpenAI (Ollama tool calling is unreliable)

    # Build full conversation
    messages = (
        [{"role": "system", "content": system_message}]
        + [{"role": h["role"], "content": h["content"]} for h in history]
        + [{"role": "user",   "content": message}]
    )

    # First call (with tools if GPT)
    kwargs   = {"tools": tools} if use_tools else {}
    response = client.chat.completions.create(model=model_id, messages=messages, **kwargs)

    # Tool calling loop: keep executing tools until the model returns a plain response
    while use_tools and response.choices[0].finish_reason == "tool_calls":
        tool_msg     = response.choices[0].message
        tool_results = handle_tool_calls(tool_msg)
        messages.append(tool_msg)
        messages.extend(tool_results)
        response = client.chat.completions.create(model=model_id, messages=messages, **kwargs)

    # Stream the final textual answer
    stream = client.chat.completions.create(model=model_id, messages=messages, stream=True)
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

print("chat() callback ready.")

chat() callback ready.


## Cell 9 – Gradio Chat UI

`gr.ChatInterface` with:
- Model selector dropdown (additional_inputs)
- Streaming enabled (chat() is a generator)
- Example questions pre-loaded

In [ ]:
model_dropdown = gr.Dropdown(
    choices=["GPT (gpt-4.1-mini)", "Ollama (llama3.2)"],
    value="GPT (gpt-4.1-mini)",
    label="Model",
    info="GPT supports tool calling (snippet library). Ollama is free & local."
)

gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Python Technical Q&A Assistant",
    description=(
        "Ask any Python or programming question. "
        "With GPT, you can also save and retrieve code snippets. "
        "Try: *'Save that snippet as fizzbuzz'* or *'What snippets do I have?'*"
    ),
    additional_inputs=[model_dropdown],
    examples=[
        ["Explain what `yield from` does in Python with an example.", "GPT (gpt-4.1-mini)"],
        ["What is the difference between a list and a generator?", "GPT (gpt-4.1-mini)"],
        ["Show me a clean implementation of FizzBuzz and save it as 'fizzbuzz'.", "GPT (gpt-4.1-mini)"],
        ["What snippets do I have saved?", "GPT (gpt-4.1-mini)"],
        ["Explain decorators in Python.", "Ollama (llama3.2)"],
    ],
    flagging_mode="never",
).launch(inbrowser=True)

## Cell 10 – Bonus: Audio Output (TTS)

A `gr.Blocks`-based UI that adds spoken audio output alongside the chat.
Uses `openai.audio.speech.create` (same pattern as Day 5).

> **Note:** Each TTS call costs a small amount. GPT model only.

In [18]:
def talker(text: str) -> bytes:
    """Convert text to speech and return raw audio bytes."""
    response = openai_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",   # alternatives: alloy, coral, echo, fable, nova, shimmer
        input=text[:4096]  # cap length to avoid very long TTS calls
    )
    return response.content


def chat_with_audio(history):
    """Chat callback for the audio UI – always uses GPT, speaks the reply.
    The user message is already the last item in history (appended by add_user_message).
    """
    messages = (
        [{"role": "system", "content": system_message}]
        + [{"role": h["role"], "content": h["content"]} for h in history]
    )

    # Tool calling loop
    response = openai_client.chat.completions.create(
        model=GPT_MODEL, messages=messages, tools=tools
    )
    while response.choices[0].finish_reason == "tool_calls":
        tool_msg     = response.choices[0].message
        tool_results = handle_tool_calls(tool_msg)
        messages.append(tool_msg)
        messages.extend(tool_results)
        response = openai_client.chat.completions.create(
            model=GPT_MODEL, messages=messages, tools=tools
        )

    # Get final reply text
    reply = response.choices[0].message.content
    history = history + [{"role": "assistant", "content": reply}]

    # Speak the reply (strip markdown for cleaner audio)
    plain_text = reply.replace("```python", "").replace("```", "").replace("**", "").replace("*", "")
    audio = talker(plain_text)

    return history, audio


def add_user_message(message, history):
    return "", history + [{"role": "user", "content": message}]


with gr.Blocks() as audio_ui:
    gr.Markdown("## Python Q&A Assistant – with Audio Output (GPT + TTS)")
    gr.Markdown(
        "Ask a Python question and hear the answer read aloud. "
        "Snippet tools are active — try *'Save a fizzbuzz snippet'*."
    )
    with gr.Row():
        chatbot = gr.Chatbot(height=450, type="messages", label="Conversation")
    with gr.Row():
        audio_output = gr.Audio(autoplay=True, label="Assistant voice")
    with gr.Row():
        msg_box = gr.Textbox(
            placeholder="Ask a Python question... (press Enter or click Send)",
            label="Your question",
            lines=1,          # lines=1 means Enter submits; lines>1 inserts a newline
        )

    # Wire both Enter key and Send button to the same two-step chain
    trigger = dict(
        fn=add_user_message,
        inputs=[msg_box, chatbot],
        outputs=[msg_box, chatbot],
    )
    followup = dict(
        fn=chat_with_audio,
        inputs=chatbot,
        outputs=[chatbot, audio_output],
    )

    msg_box.submit(**trigger).then(**followup)

audio_ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
